# [RAGAs를 이용한 데이터셋 생성](https://docs.ragas.io/en/stable/howtos/customizations/testgenerator/_language_adaptation/#generate)

## Load Dataset

In [1]:
import pandas as pd 

df = pd.read_csv("./data/cleaning/illnesses.csv")

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head(2)

(전체 데이터 수, 전체 컬럼 수): (129, 4)


,video_url,title,clean_caption,title_caption
0,https://www.youtube.com/watch?v=8GwNlF44ntY&li...,식은땀! 이 정도는 알아두셔야 합니다.,"안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가...","[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요,..."
1,https://www.youtube.com/watch?v=ZO1ffVTWsJc,"식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다","안녕하세요, 소아청소년과 전문의입니다. 날이 더운 여름에는 식중독이 많이 발생합니다...","[title] 식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다 [caption]..."


## Loader

In [2]:
from langchain_community.document_loaders import CSVLoader

# illnesses.csv -> caption만 로드 진행할 수 있는 객체
loader = CSVLoader(
    file_path="./data/cleaning/illnesses.csv" # 파일명 
    , source_column='video_url' # 소스 컬럼 
    , content_columns=['title_caption'] # 컨텐츠(학습) 컬럼 
    , encoding="utf-8"
)

In [3]:
docs = loader.load() # load()를 이용해서 데이터 로드 진행..

In [4]:
# df.shape -> (데이터의 수(=유튜브 영상의 수), 컬럼의 수)
len(docs), df.shape

(129, (129, 4))

In [5]:
docs[0].metadata['source']

'https://www.youtube.com/watch?v=8GwNlF44ntY&list=PLC6Msm1YCNw-NJBk4ajBkTeS2rJULiT1b'

In [6]:
docs[0].page_content

'title_caption: [title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가 땀을 많이 흘리는데 어디가 약한 건 아닌가 걱정하시는 분들이 종종 있습니다. 감기약을 먹고 있으면 약이 너무 센 것은 아닌가라고 물어보기도 합니다. 이번에는 많은 부모들이 고민하는 식은땀에 대해 말씀드리겠습니다. 사람은 원래 땀이 나게 되어 있고, 사람 몸에는 약 200만 개의 땀샘이 있어 땀을 흘리게 됩니다. 땀은 우리 몸의 노폐물을 배출하고 체온을 조절하는 역할을 하므로 적당히 땀을 흘리는 것은 정상입니다. 사람마다 차이는 크지만 아이들은 어른에 비해 땀을 더 많이 흘리는 편입니다. 게다가 아이들은 땀을 조절하는 기능이 어른보다 미숙하기 때문에 이마, 손바닥, 발바닥 등은 평소에도 쉬거나 조금만 힘을 써도 땀이 송골송골 맺히는 경우가 흔합니다. 별다른 문제가 없어도 베개가 땀으로 젖는 경우도 있습니다. 물론 감기에 걸리거나 다른 이상이 있는 경우에는 땀 조절 기능이 더 떨어져 땀을 과잉 생산하기도 합니다. 아이들은 원래 땀을 많이 흘리는 편인데, 그렇다고 해서 반드시 몸이 허약한 것은 아닙니다. 아기가 아프지 않은데도 땀을 많이 흘리니 몸이 허약하다며 보양식을 챙겨주거나 영양제를 먹이는 부모도 많습니다. 하지만 부모들이 걱정해서 소아청소년과에 데려와도 대개 심각한 병은 아닌 경우가 많습니다. 아이가 땀을 많이 흘려도 열이 없고 겉으로 보기에 다른 이상이 없다면 대개 정상적이라고 볼 수 있습니다. 특히 아기가 어릴수록 땀을 많이 흘리고, 갑자기 평소보다 더 많이 흘리는 경우도 있습니다. 특히 신생아는 식은땀을 많이 흘리는 편입니다. 다만 간혹 진짜 문제가 있는 경우도 있습니다. 선천성 심장병이나 갑상선 기능 항진증처럼 체력을 소모시키는 병에 걸리면 땀을 많이 흘릴 수 있습니다. 구루병, 저혈당, 핵황달 같은 심각한 병에 걸려도 땀을 많이 흘릴 수 있지만 이런 병들은 원래 매우 

## Model

### [OpenAI API Key](https://platform.openai.com/settings/organization/api-keys)

In [7]:
from dotenv import load_dotenv 

# .env 파일에 있는 데이터를 환경변수에 등록해주는 함수 
load_dotenv()

True

## 학습 데이터셋

### [문장을 생성하는 LLM](https://platform.openai.com/docs/models)

In [8]:
# Langchain LLM 객체를 ragas가 인식할 수 있게 변형
# Wrapper -> 감싸다. 또는 형변환 된다...
from ragas.llms import LangchainLLMWrapper
# Langchain에서 인식할 수 있는 OpenaAI LLM 객체
from langchain_openai import ChatOpenAI

# 문장을 생성하는 LLM -> 질문 & 답변 생성을 목적으로...
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

C:\Users\Playdata\AppData\Local\Temp\ipykernel_21324\211073525.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(


### [Embedding model](https://platform.openai.com/docs/guides/embeddings/embedding-models#embedding-models)

In [9]:
# RAGAs의 최신 OpenAIEmbeddings 사용 (client 필요)
from ragas.embeddings import OpenAIEmbeddings
from openai import OpenAI

# Embedding model -> 문장을 벡터로 변환하는 모델...
generator_embeddings = OpenAIEmbeddings(
    # OpenAI client 생성 (환경변수의 OPENAI_API_KEY를 자동으로 사용)
    client=OpenAI(),
    model="text-embedding-3-small"
)

### 페르소나 정의(중요)
> 어떤 질문/답변 쌍을 만들지 정의함 

In [10]:
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="a pediatrician", # 소아과 의사 선생님
        # 일반 대중이 이해하기 쉽게 질병을 설명해주는 의사
        role_description=""""
        You are a pediatrician who explains childhood and toddler illnesses clearly, accurately, and in an easy-to-understand way for parents and caregivers.
        """,
    ),
    Persona(
        name="a compassionate pediatrician", # 소아과 의사 선생님
        # 부모 불안 완화
        role_description=""""
        You are a compassionate pediatrician who helps parents understand toddler illnesses using clear, reassuring explanations while emphasizing when to seek medical care.
        """,
    ),
    Persona(
        name="a pediatric medical expert", # 소아과 의사 선생님
        # 원인·증상·대처 중심
        role_description=""""
        You are a pediatric medical expert who educates parents and caregivers about common childhood illnesses, their causes, symptoms, and appropriate home care in a clear and structured manner.
        """,
    ),
]

### 학습용 데이터를 만드는 객체

In [11]:
from ragas.testset import TestsetGenerator

# 학습용 데이터를 만드는 객체
generator = TestsetGenerator(
    llm=generator_llm, embedding_model=generator_embeddings, 
    persona_list=personas
)

NERExtractor: 
> RAG 평가용 테스트셋 생성 시, 문서에 등장하는 핵심 엔티티를 뽑아 질문을 자동 생성하는 클래스입니다.

In [12]:
from ragas.testset.transforms.extractors.llm_based import NERExtractor

transforms = [NERExtractor(llm=generator_llm)]

SingleHopSpecificQuerySynthesizer
> Ragas의 테스트셋 생성 기능에서, 단일 문서 기반(single-hop)으로 “특정하고 구체적인 질문”을 자동 생성하는 Synthesizer 클래스입니다.

In [13]:
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]

for query, _ in distribution:
    prompts = await query.adapt_prompts("korean", llm=generator_llm)
    query.set_prompts(**prompts)

### 학습용 데이터셋 생성

In [14]:
dataset = generator.generate_with_langchain_docs(
    docs[:],
    testset_size=800, # 총 몇개의 질문/답변 쌍을 만들지
    transforms=transforms,
    query_distribution=distribution,
)

Applying NERExtractor:   0%|          | 0/129 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/805 [00:00<?, ?it/s]

## 생성된 데이터셋 확인

In [15]:
eval_dataset = dataset.to_evaluation_dataset()

In [16]:
print("Query:", eval_dataset[0].user_input)
print("Reference:", eval_dataset[0].reference)

Query: 선천성 심장병이 아이가 땀을 많이 흘리는 것과 어떤 관계가 있나요? 그리고 부모들은 이걸 어떻게 이해해야 하나요?
Reference: 선천성 심장병은 체력을 소모시키는 병으로, 이 병에 걸리면 아이가 땀을 많이 흘릴 수 있습니다. 그러나 땀만으로 그런 병이라고 판단하기는 어렵고, 대개 다른 증상이 함께 있어야 의심합니다. 부모들은 아이가 땀을 많이 흘려도 열이 없고 겉으로 보기에 다른 이상이 없다면 대개 정상적이라고 볼 수 있으며, 너무 걱정하지 말고 수분을 충분히 섭취시키고 땀을 잘 닦아주며 옷을 자주 갈아입혀 주면 마음이 편해질 것입니다.


In [17]:
# 생성된 테스트셋을 pandas DataFrame으로 변환
eval_df = eval_dataset.to_pandas()
eval_df.shape

(805, 6)

In [18]:
eval_df.head(2)

,user_input,reference_contexts,reference,persona_name,query_style,query_length
0,선천성 심장병이 아이가 땀을 많이 흘리는 것과 어떤 관계가 있나요? 그리고 부모들은...,[title_caption: [title] 식은땀! 이 정도는 알아두셔야 합니다. ...,"선천성 심장병은 체력을 소모시키는 병으로, 이 병에 걸리면 아이가 땀을 많이 흘릴 ...",a pediatrician,POOR_GRAMMAR,LONG
1,소아청소년과에서 아이 땀 많이 흘리는거 괜찮은거야?,[title_caption: [title] 식은땀! 이 정도는 알아두셔야 합니다. ...,"아이들이 땀을 많이 흘리는 것은 일반적이며, 이는 정상적인 현상입니다. 아이들은 어...",a pediatrician,POOR_GRAMMAR,SHORT


> Persona

In [19]:
eval_df['persona_name'].unique()

array(['a pediatrician', 'a compassionate pediatrician',
       'a pediatric medical expert'], dtype=object)

> Query style

In [20]:
eval_df['query_style'].unique()

array(['POOR_GRAMMAR', 'MISSPELLED', 'PERFECT_GRAMMAR', 'WEB_SEARCH_LIKE'],
      dtype=object)

> Query length

In [21]:
eval_df['query_length'].unique()

array(['LONG', 'SHORT', 'MEDIUM'], dtype=object)

### reference_video_url 추가하기 

In [22]:
eval_df.loc[0,['reference_contexts']].values[0][0].replace("title_caption: ", "")

'[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가 땀을 많이 흘리는데 어디가 약한 건 아닌가 걱정하시는 분들이 종종 있습니다. 감기약을 먹고 있으면 약이 너무 센 것은 아닌가라고 물어보기도 합니다. 이번에는 많은 부모들이 고민하는 식은땀에 대해 말씀드리겠습니다. 사람은 원래 땀이 나게 되어 있고, 사람 몸에는 약 200만 개의 땀샘이 있어 땀을 흘리게 됩니다. 땀은 우리 몸의 노폐물을 배출하고 체온을 조절하는 역할을 하므로 적당히 땀을 흘리는 것은 정상입니다. 사람마다 차이는 크지만 아이들은 어른에 비해 땀을 더 많이 흘리는 편입니다. 게다가 아이들은 땀을 조절하는 기능이 어른보다 미숙하기 때문에 이마, 손바닥, 발바닥 등은 평소에도 쉬거나 조금만 힘을 써도 땀이 송골송골 맺히는 경우가 흔합니다. 별다른 문제가 없어도 베개가 땀으로 젖는 경우도 있습니다. 물론 감기에 걸리거나 다른 이상이 있는 경우에는 땀 조절 기능이 더 떨어져 땀을 과잉 생산하기도 합니다. 아이들은 원래 땀을 많이 흘리는 편인데, 그렇다고 해서 반드시 몸이 허약한 것은 아닙니다. 아기가 아프지 않은데도 땀을 많이 흘리니 몸이 허약하다며 보양식을 챙겨주거나 영양제를 먹이는 부모도 많습니다. 하지만 부모들이 걱정해서 소아청소년과에 데려와도 대개 심각한 병은 아닌 경우가 많습니다. 아이가 땀을 많이 흘려도 열이 없고 겉으로 보기에 다른 이상이 없다면 대개 정상적이라고 볼 수 있습니다. 특히 아기가 어릴수록 땀을 많이 흘리고, 갑자기 평소보다 더 많이 흘리는 경우도 있습니다. 특히 신생아는 식은땀을 많이 흘리는 편입니다. 다만 간혹 진짜 문제가 있는 경우도 있습니다. 선천성 심장병이나 갑상선 기능 항진증처럼 체력을 소모시키는 병에 걸리면 땀을 많이 흘릴 수 있습니다. 구루병, 저혈당, 핵황달 같은 심각한 병에 걸려도 땀을 많이 흘릴 수 있지만 이런 병들은 원래 매우 드물고 대개 다른 증상이 함

In [23]:
# df[['video_url', 'title_caption']].head(2)
df.loc[0,['title_caption']].values[0]

'[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가 땀을 많이 흘리는데 어디가 약한 건 아닌가 걱정하시는 분들이 종종 있습니다. 감기약을 먹고 있으면 약이 너무 센 것은 아닌가라고 물어보기도 합니다. 이번에는 많은 부모들이 고민하는 식은땀에 대해 말씀드리겠습니다. 사람은 원래 땀이 나게 되어 있고, 사람 몸에는 약 200만 개의 땀샘이 있어 땀을 흘리게 됩니다. 땀은 우리 몸의 노폐물을 배출하고 체온을 조절하는 역할을 하므로 적당히 땀을 흘리는 것은 정상입니다. 사람마다 차이는 크지만 아이들은 어른에 비해 땀을 더 많이 흘리는 편입니다. 게다가 아이들은 땀을 조절하는 기능이 어른보다 미숙하기 때문에 이마, 손바닥, 발바닥 등은 평소에도 쉬거나 조금만 힘을 써도 땀이 송골송골 맺히는 경우가 흔합니다. 별다른 문제가 없어도 베개가 땀으로 젖는 경우도 있습니다. 물론 감기에 걸리거나 다른 이상이 있는 경우에는 땀 조절 기능이 더 떨어져 땀을 과잉 생산하기도 합니다. 아이들은 원래 땀을 많이 흘리는 편인데, 그렇다고 해서 반드시 몸이 허약한 것은 아닙니다. 아기가 아프지 않은데도 땀을 많이 흘리니 몸이 허약하다며 보양식을 챙겨주거나 영양제를 먹이는 부모도 많습니다. 하지만 부모들이 걱정해서 소아청소년과에 데려와도 대개 심각한 병은 아닌 경우가 많습니다. 아이가 땀을 많이 흘려도 열이 없고 겉으로 보기에 다른 이상이 없다면 대개 정상적이라고 볼 수 있습니다. 특히 아기가 어릴수록 땀을 많이 흘리고, 갑자기 평소보다 더 많이 흘리는 경우도 있습니다. 특히 신생아는 식은땀을 많이 흘리는 편입니다. 다만 간혹 진짜 문제가 있는 경우도 있습니다. 선천성 심장병이나 갑상선 기능 항진증처럼 체력을 소모시키는 병에 걸리면 땀을 많이 흘릴 수 있습니다. 구루병, 저혈당, 핵황달 같은 심각한 병에 걸려도 땀을 많이 흘릴 수 있지만 이런 병들은 원래 매우 드물고 대개 다른 증상이 함

In [24]:
eval_df['reference_title_caption'] = eval_df['reference_contexts'].map(
    lambda x: x[0].replace("title_caption: ", "").strip()
)

eval_df[['reference_title_caption', 'reference_contexts']].head(2)

,reference_title_caption,reference_contexts
0,"[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요,...",[title_caption: [title] 식은땀! 이 정도는 알아두셔야 합니다. ...
1,"[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요,...",[title_caption: [title] 식은땀! 이 정도는 알아두셔야 합니다. ...


In [25]:
eval_df = eval_df.merge(
    right=df[['video_url', 'title_caption']], how='inner',
    left_on="reference_title_caption", right_on="title_caption"
)

In [26]:
eval_df.rename(
    columns={
        "video_url": "reference_video_url"
    }, inplace= True
)

In [27]:
eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']].head()

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,선천성 심장병이 아이가 땀을 많이 흘리는 것과 어떤 관계가 있나요? 그리고 부모들은...,"선천성 심장병은 체력을 소모시키는 병으로, 이 병에 걸리면 아이가 땀을 많이 흘릴 ...",https://www.youtube.com/watch?v=8GwNlF44ntY&li...,a pediatrician,POOR_GRAMMAR,LONG
1,소아청소년과에서 아이 땀 많이 흘리는거 괜찮은거야?,"아이들이 땀을 많이 흘리는 것은 일반적이며, 이는 정상적인 현상입니다. 아이들은 어...",https://www.youtube.com/watch?v=8GwNlF44ntY&li...,a pediatrician,POOR_GRAMMAR,SHORT
2,구루병이 아이의 땀과 어떤 관련이 있나요?,"구루병은 심각한 병 중 하나로, 땀을 많이 흘릴 수 있지만 이러한 병들은 원래 매우...",https://www.youtube.com/watch?v=8GwNlF44ntY&li...,a compassionate pediatrician,MISSPELLED,SHORT
3,소아청소년과에서 아이의 식은땀에 대해 어떻게 설명하나요?,소아청소년과 전문의는 아이들이 땀을 많이 흘리는 것은 정상적이라고 설명합니다. 아이...,https://www.youtube.com/watch?v=8GwNlF44ntY&li...,a compassionate pediatrician,PERFECT_GRAMMAR,SHORT
4,핵황달이 땀과 관련이 있나요?,"핵황달은 심각한 병 중 하나로, 땀을 많이 흘릴 수 있지만 이런 병들은 원래 매우 ...",https://www.youtube.com/watch?v=8GwNlF44ntY&li...,a pediatric medical expert,POOR_GRAMMAR,SHORT


In [28]:
eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']].tail()

,user_input,reference,reference_video_url,persona_name,query_style,query_length
800,응급실에 가야하는 아이의 증상은 어떤 것들이 있나요?,아이의 응급실 방문이 필요할 수 있는 위험한 증상으로는 숨쉬기 힘들거나 호흡이 매우...,https://www.youtube.com/watch?v=rizkolEXRY4,a pediatric medical expert,MISSPELLED,MEDIUM
801,응급실에 가야 할 위험한 증상은 무엇인가요?,아이에게 코로나에 걸렸을 때 즉시 119에 연락하거나 병원을 찾아야 할 위험한 증상...,https://www.youtube.com/watch?v=rizkolEXRY4,a pediatric medical expert,WEB_SEARCH_LIKE,LONG
802,고열이 항상 위험한가요?,고열 자체가 항상 위험한 것은 아닙니다. 해열제를 먹여도 열이 금방 떨어지지 않는 ...,https://www.youtube.com/watch?v=rizkolEXRY4,a pediatrician,POOR_GRAMMAR,SHORT
803,응급실에 가야 할 위험한 증상은 무엇인가요?,아이들이 코로나에 걸렸을 때 즉시 119에 연락하거나 병원을 찾아야 할 위험한 증상...,https://www.youtube.com/watch?v=rizkolEXRY4,a pediatrician,WEB_SEARCH_LIKE,LONG
804,고열이 있는 아이가 코로나에 걸렸을 때 부모가 알아야 할 중요한 점은 뭐예요?,"고열 자체가 항상 위험한 것은 아니지만, 5일 이상 열이 지속되거나 38도 이상의 ...",https://www.youtube.com/watch?v=rizkolEXRY4,a compassionate pediatrician,POOR_GRAMMAR,LONG


> 중복 데이터 제거 

In [29]:
eval_df = eval_df[['user_input', 'reference', 'reference_video_url', 
    'persona_name', 'query_style', 'query_length']]
print(f"befer: {eval_df.shape}")

eval_df.drop_duplicates(inplace=True, subset=['user_input', 'reference'])
print(f"after: {eval_df.shape}")

befer: (805, 6)
after: (793, 6)


> 결측치 확인

In [30]:
eval_df.isnull().sum()

user_input             0
reference              0
reference_video_url    0
persona_name           0
query_style            0
query_length           0
dtype: int64

### 저장하기 

> 데이터를 랜덤하게 정렬하기 

- `sample(frac=1)` : 전체 행을 랜덤 셔플
- `random_state=42` : 재현 가능하도록 시드 고정 (선택)
- `reset_index(drop=True)` : 기존 index 제거 후 0부터 새로 생성

In [31]:
save_df = eval_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [32]:
save_df.head()

,user_input,reference,reference_video_url,persona_name,query_style,query_length
0,소아청소년과에서 수족구병에 대해 부모님들이 알아야 할 중요한 정보는 뭐에요?,"수족구병은 전염성이 매우 강한 질병으로, 급성기에는 어린이집이나 유치원에 보내지 않...",https://www.youtube.com/watch?v=omxS7s9dn-E,a compassionate pediatrician,POOR_GRAMMAR,LONG
1,일본뇌염 예방을 위해 IR3535 성분의 모기기피제를 사용하는 것이 왜 중요한가요?,IR3535 성분의 모기기피제를 사용하는 것은 일본뇌염 경보가 발령된 상황에서 모기...,https://www.youtube.com/watch?v=VNS2QM4M5jo,a pediatric medical expert,PERFECT_GRAMMAR,MEDIUM
2,하정우 소아청소년과 전문의가 제시한 코로나 격리 기간 동안의 쓰레기 처리 지침은 무...,"하정우 소아청소년과 전문의는 감염이 의심되는 환자는 격리하고 외출을 자제해야 하며,...",https://www.youtube.com/watch?v=Uf0uPeOasA4,a compassionate pediatrician,WEB_SEARCH_LIKE,LONG
3,코로나와 관련하여 소아청소년과에서 부모가 주의해야 할 점은 무엇인가요?,코로나 유행으로 호흡기 질환 발생이 줄어 아이들이 면역을 충분히 형성하지 못했을 가...,https://www.youtube.com/watch?v=evUMFZcLkeo,a compassionate pediatrician,WEB_SEARCH_LIKE,LONG
4,구토는 요로감염의 어떤 증상으로 나타날 수 있나요?,"구토는 요로감염의 증상 중 하나로, 특히 어린 아기들에게 나타날 수 있습니다. 이 ...",https://www.youtube.com/watch?v=eXCjsY4NvSk,a pediatric medical expert,PERFECT_GRAMMAR,SHORT


> 저장하기 

In [33]:
save_df.to_csv("./data/training/illnesses.csv", index=False, header=True)